In [ ]:
import pandas as pd              # Mengolah data
import requests                  # Mengirim HTTP request
from bs4 import BeautifulSoup    # Parsing HTML
import time                      # Delay scraping

# Mengambil halaman web
def get_soup(url, session):
    try:
        response = session.get(url, headers={'User-Agent': 'Mozilla/5.0'})   # Request halaman
        response.raise_for_status()                                           # Cek error HTTP
        return BeautifulSoup(response.text, 'html.parser')                    # Parse HTML
    except Exception as e:
        print(f"Error fetching {url}: {e}")                                   # Tampilkan error
        return None                                                           # Kembalikan None

# Mengambil isi artikel
def extract_article_content(soup, url):

    # Struktur data artikel
    data = {
        "Title": "",          # Judul
        "Date": "",           # Tanggal
        "Content": "",        # Isi artikel
        "Category": "",       # Kategori
        "URL": url            # URL artikel
    }

    try:

        # Ambil judul
        title = soup.find('h1')
        if title:
            data['Title'] = title.get_text(strip=True)

        # Ambil tanggal
        date = soup.find('div', class_='text-cnn_grey text-sm mb-4')
        if date:
            data['Date'] = date.get_text(strip=True)

        # Ambil konten utama
        content_div = soup.find('div', class_='detail-text text-cnn_black text-sm grow min-w-0')
        if content_div:

            # Hapus blok iklan
            for ad in content_div.find_all('div', class_='paradetail'):
                ad.decompose()

            # Ambil seluruh paragraf
            paragraphs = content_div.find_all('p')
            clean_paras = []

            # Bersihkan paragraf
            for p in paragraphs:
                classes = p.get('class', [])

                # Lewati caption
                if 'para_caption' in classes:
                    continue

                # Ambil teks paragraf
                text = p.get_text(" ", strip=True)

                # Lewati teks iklan
                if text.upper() in ["ADVERTISEMENT", "SCROLL TO CONTINUE WITH CONTENT"]:
                    continue

                # Simpan paragraf
                if text:
                    clean_paras.append(text)

            # Gabungkan isi artikel
            data['Content'] = ' '.join(clean_paras)

        # Ambil kategori
        breadcrumb = soup.find('a', class_='gtm_breadcrumb_subkanal')
        if breadcrumb:
            data['Category'] = breadcrumb.get_text(strip=True)

    except Exception as e:
        print(f"Error processing article {url}: {e}")     # Error parsing

    return data                                            # Kembalikan data

# Scraping halaman kategori
def scrape_category_page(main_url, max_articles=15):

    session = requests.Session()           # Session request
    all_article_data = []                  # Menyimpan artikel
    page_num = 1                           # Halaman awal

    # Selama target belum tercapai
    while len(all_article_data) < max_articles:

        current_url = f"{main_url}?page={page_num}"        # URL halaman
        print(f"Scraping page: {current_url}")

        soup = get_soup(current_url, session)              # Ambil halaman
        if not soup:
            break

        # Cari daftar artikel
        articles = soup.find_all('article', class_='flex-grow')
        article_urls = []

        # Ambil URL artikel
        for article in articles:
            a_tag = article.find('a')
            if a_tag and a_tag.has_attr('href'):
                url = a_tag['href']

                # Lewati URL tidak valid
                if url == '#' or 'video' in url:
                    continue

                # Lengkapi URL relatif
                if not url.startswith('http'):
                    url = f"https://www.cnnindonesia.com{url}"

                article_urls.append(url)

        # Berhenti jika kosong
        if not article_urls:
            print(f"No articles found on page {page_num}. Stopping.")
            break

        # Proses setiap artikel
        for url in article_urls:

            # Cek jumlah artikel
            if len(all_article_data) >= max_articles:
                break

            print(f"Processing article: {url}")

            article_soup = get_soup(url, session)      # Ambil artikel

            if article_soup:

                article_data = extract_article_content(article_soup, url)

                # Pastikan isi tidak kosong
                if article_data['Content']:

                    # Hitung jumlah kata
                    word_count = len(article_data['Content'].split())

                    # Seleksi 250–350 kata
                    if 250 <= word_count <= 350:
                        all_article_data.append(article_data)
                        print(f"Artikel ditambahkan. Total artikel saat ini: {len(all_article_data)}")
                    else:
                        print(f"Konten kurang dari 250 atau lebih dari 350 kata ({word_count} kata), dilewati.")

                else:
                    print("Konten kosong, lewati dan tambahkan satu ke target artikel.")

            time.sleep(1)      # Delay artikel

        page_num += 1          # Halaman berikutnya
        time.sleep(2)          # Delay halaman

    return all_article_data    # Hasil scraping

# Program utama
if __name__ == "__main__":

    # Daftar kategori
    kategori_list = [
        {"url": "https://www.cnnindonesia.com/info-politik/indeks/632"}
    ]

    all_data = []      # Menyimpan seluruh data

    # Scraping setiap kategori
    for category in kategori_list:
        print(f"Memulai scraping untuk kategori: {category['url']}")
        category_data = scrape_category_page(category['url'], max_articles=15)
        all_data.extend(category_data)

    # Konversi ke DataFrame
    df = pd.DataFrame(all_data)

    # Simpan CSV
    df.to_csv('cnn_infopolitik_data.csv', index=False)

    # Simpan Excel
    df.to_excel('cnn_infopolitik_data.xlsx', index=False)

    # Informasi selesai
    print("Scraping selesai. Data disimpan dalam cnn_infopolitik_data.csv")

Memulai scraping untuk kategori: https://www.cnnindonesia.com/info-politik/indeks/632
Scraping page: https://www.cnnindonesia.com/info-politik/indeks/632?page=1
Processing article: https://www.cnnindonesia.com/nasional/20260621130821-633-1371451/said-dorong-kebijakan-afirmatif-tarif-cukai-rokok-golongan-iii
Konten kurang dari 250 atau lebih dari 350 kata (629 kata), dilewati.
Processing article: https://www.cnnindonesia.com/nasional/20260618193337-633-1370596/said-tegaskan-posisi-pdip-sebagai-partai-penyeimbang-pemerintah
Artikel ditambahkan. Total artikel saat ini: 1
Processing article: https://www.cnnindonesia.com/nasional/20260614172432-633-1369052/dasco-apresiasi-langkah-baru-bank-indonesia-perkuat-rupiah
Konten kurang dari 250 atau lebih dari 350 kata (222 kata), dilewati.
Processing article: https://www.cnnindonesia.com/nasional/20260611153755-638-1367974/kwp-dan-bni-salurkan-2000-paket-alat-tulis-jelang-tahun-ajaran-baru
Konten kurang dari 250 atau lebih dari 350 kata (411 kata)